# Mouse skull transparency — **TIPS at 1 MHz on the cerebellum** — Colab

The mouse counterpart of the [human 500 kHz notebook](https://github.com/pinton-lab/skull_transparency/blob/main/notebooks/skull_transparency_colab_500kHz.ipynb). It runs the same pipeline end to end on a real mouse skull and produces the **same placement report** as `examples/mouse_tips_cerebellum` — anatomy with the Allen cerebellum, the transparency map, the chosen window, and (optionally) a transcranial-vs-free-field comparison — as a PDF you download at the end.

**Pipeline:** fetch skull + atlas → `prepare` (source at the cerebellum) → GPU solve → `extract` → transparency + placement → report.

**Skull:** the Maga (University of Washington) 4K dry-skull microCT, staged through the TUBA mouse pipeline: a 200 µm sound-speed volume in the skull's own RAS frame, plus the Allen CCFv3 annotation warped into that same frame (\~7 MB in total, fetched below).

**Transducer:** TIPS (Philips Therapeutic Imaging Probe and Sonication) — an 80 mm radius of curvature, 20.5–46 mm annular aperture, driven at **1 MHz**.

**Two things a mouse changes.** Points per wavelength is set by **bone thickness**, not wavelength: 6 ppw at 1 MHz gives dx = 0.257 mm against a \~0.2–0.5 mm calvaria (one voxel), so this runs **12 ppw** (dx = 0.128 mm). And the placement **footprint** comes from the beam cone's half-angle, not the aperture radius — the mouse skull sits \~6 mm from the cerebellum while the TIPS focal length is 80 mm, so the cone crosses it at 3.4 mm, not 46 mm. Both are explained where they are used.

## Setup

**Set the runtime to GPU first:** *Runtime ▸ Change runtime type ▸ GPU (T4)*, then run the cells top to bottom. The whole notebook takes a few minutes.

In [ ]:
# --- install the pipeline + fetch the CUDA solver binary ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules

# the mouse case lives on the `mouse` branch (report anatomy/forward sections included);
# --force-reinstall --no-deps replaces any cached copy from an earlier session.
!pip -q install --force-reinstall --no-deps "git+https://github.com/pinton-lab/skull_transparency.git@mouse"
!pip -q install nibabel
_help = subprocess.run(["skull-transparency", "prepare", "--help"], capture_output=True, text=True).stdout
assert "--center-mm" in _help, ("A stale skull_transparency is active. Fix: Runtime > Disconnect and "
    "delete runtime, then reopen this notebook from GitHub and Run all.")
print("skull_transparency ready")

!git clone --depth 1 https://github.com/pinton-lab/fullwave2-ultra.git /content/fullwave2-ultra 2>/dev/null || true
hits = glob.glob("/content/fullwave2-ultra/**/bench_3d_opt", recursive=True)   # locate the binary
assert hits, "bench_3d_opt not found in the clone -- re-run this cell (network?) or check the repo."
os.environ["FULLWAVE2_BIN"] = hits[0]
os.chmod(hits[0], 0o755)                                                      # make it executable
print("solver:", os.environ["FULLWAVE2_BIN"])

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE VISIBLE -- Runtime > Change runtime type > GPU")

### 1 · Fetch the mouse skull and the atlas

Two small files, both in the skull's own aligned-native RAS frame: the **sound-speed source volume** (200 µm) and the **Allen CCFv3 annotation** warped into that frame, which supplies the cerebellum. If `MOUSE_DATA_URL` is not set, point the paths at your own copies (e.g. a mounted Drive folder).

In [ ]:
import os, urllib.request, hashlib

#: where the two staged inputs live. Set this to the published location (or a Drive path).
MOUSE_DATA_URL = os.environ.get("MOUSE_DATA_URL", "")     # e.g. "https://.../mouse_inputs_v1"
FILES = {"maga_skull_aligned_native_200um.nii.gz":
         "b3eadf213abdd7d88ac15f188ffe450717056cb33f4d09209d17b09ed25ebd75",
         "allen_annotation_in_maga.nii.gz":
         "6bbb3524cf5c831e13240de731e83deae32b7abbcc94f9c95f655cce92158ecd"}

def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

for name, want in FILES.items():
    if not os.path.exists(name):
        if not MOUSE_DATA_URL:
            raise SystemExit(
                f"{name} is not here and MOUSE_DATA_URL is unset. Set MOUSE_DATA_URL to the "
                "location of the two staged inputs, or upload them next to this notebook "
                "(they are ~7 MB together).")
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(f"{MOUSE_DATA_URL.rstrip('/')}/{name}", name)
    got = _sha256(name)
    print(f"  {name}  {'OK' if got == want else 'CHECKSUM MISMATCH'}")
    assert got == want, f"{name} does not match the recorded checksum -- re-download."
SKULL_NII, ANNOT_NII = list(FILES)

### 2 · Build the medium and locate the cerebellum

Hounsfield-like microCT intensity is ramped to sound speed with the same map the rest of the mouse work uses (5000 counts → water 1540 m/s, 25000 → cortical bone 2900 m/s). The **target** is the centroid of the Allen cerebellum — every descendant of `CB` (id 512) in the Allen structure graph — computed from the warped annotation, so it is derived here rather than hard-coded.

In [ ]:
import numpy as np, nibabel as nib

I_LOW, I_HIGH = 5000.0, 25000.0          # maga intensity -> sound-speed ramp
C_WATER, C_BONE_MAX = 1540.0, 2900.0
BONE_THRESHOLD = 1800.0                  # m/s; calvarial-surface cutoff for this c-map
INPUT_FRAME = "maga_aligned_native_ras_mm"

#: Allen CCFv3 cerebellum: every descendant of CB (id 512), embedded so this needs no network.
CEREBELLUM_IDS = (
    91, 512, 519, 528, 645, 846, 912, 920, 928, 936, 944, 951,
    957, 968, 976, 984, 989, 992, 1001, 1007, 1017, 1025, 1033, 1041,
    1049, 1056, 1064, 1073, 1091, 1143, 1144, 1145, 10672, 10673, 10674, 10675,
    10676, 10677, 10678, 10679, 10680, 10681, 10682, 10683, 10684, 10685, 10686, 10687,
    10688, 10689, 10690, 10691, 10692, 10705, 10706, 10707, 10708, 10709, 10710, 10711,
    10712, 10713, 10714, 10715, 10716, 10717, 10718, 10719, 10720, 10721, 10722, 10723,
    10724, 10725, 10726, 10727, 10728, 10729, 10730, 10731, 10732, 10733, 10734, 10735,
    10736, 10737, 589508455)

img = nib.load(SKULL_NII)
affine = np.asarray(img.affine, float)
_t = np.clip((np.asarray(img.dataobj, dtype=np.float32) - I_LOW) / (I_HIGH - I_LOW), 0, 1)
c_map = (C_WATER + _t * (C_BONE_MAX - C_WATER)).astype(np.float32)

ann = nib.load(ANNOT_NII)
A = np.asarray(ann.affine, float)
mask = np.isin(np.asarray(ann.dataobj).astype(np.int64), np.asarray(CEREBELLUM_IDS))
idx = np.array(np.nonzero(mask), float)
TARGET = A[:3, :3] @ idx.mean(1) + A[:3, 3]
print(f"c-map {c_map.shape} @ {abs(affine[0,0])*1e3:.0f} um   bone {int((c_map>BONE_THRESHOLD).sum())} voxels")
print(f"cerebellum: {int(mask.sum())} voxels = {mask.sum()*abs(np.linalg.det(A[:3,:3])):.1f} mm^3")
print(f"target (cerebellum centroid): {np.round(TARGET, 2)} mm  [{INPUT_FRAME}]")

### 3 · `prepare` + the outward GPU solve

One omnidirectional source at the cerebellum radiates out through the skull; the surface-shell recorder keeps only the calvarial field, which is all the transparency map needs. The grid is \~250 × 290 × 210 at 0.128 mm and the solve takes seconds.

In [ ]:
from skull_transparency.transducer_spec import TransducerSpec
from skull_transparency.sim.prepare import build_brain_center_run
from skull_transparency.sim.launchers import launch_outward
from skull_transparency.sim.extract import extract_bundle
import json, pathlib

#: TIPS: 80 mm radius of curvature, 92 mm aperture (20.5-46 mm annulus), 8 rings, at 1 MHz.
#: ppw 12 -- set by the ~0.3 mm mouse calvaria, not by the 1.54 mm wavelength.
TIPS = TransducerSpec(f0_hz=1e6, geometry="annular", roc_mm=80.0, aperture_mm=92.0, n_rings=8,
                      c0_ms=C_WATER, ppw=12.0, acceptance_angle_deg=35.0)

SIM = pathlib.Path("run_mouse")
sim = build_brain_center_run(c_map, affine, TIPS, SIM, center_phys_mm=TARGET,
                             bone_threshold=BONE_THRESHOLD, surround_mm=6.0,
                             input_frame=INPUT_FRAME)
meta = json.loads((pathlib.Path(sim) / "meta.json").read_text())
gs = meta.get("grid_shape", [meta["N"]] * 3)
print(f"grid {gs[0]}x{gs[1]}x{gs[2]}  dx {TIPS.dx_mm:.3f} mm ({TIPS.ppw:.0f} ppw at 1 MHz)")

outdir = launch_outward(str(sim), str(SIM), run_solver=True, recorder="shell")
assert (pathlib.Path(outdir) / "SUCCESS").exists(), "solver did not finish -- see the log above"
bundle_dir = extract_bundle(SIM / "outward", SIM / "bundle", SIM, bone_threshold=BONE_THRESHOLD)
print("Field Bundle:", bundle_dir)

### 4 · Transparency map and TIPS placement

The window search aggregates the incidence-weighted coupling over the **beam cone's** footprint on the skull. Using the TIPS aperture radius (46 mm) instead would swallow the whole 25 mm head and flatten the score — so the footprint is derived from the half-angle and the skull's actual distance from the target.

In [ ]:
import skull_transparency as st

bundle = st.load_bundle(bundle_dir)
tmap = st.compute_transparency_map(bundle)

r_skull = float(np.percentile(np.asarray(tmap.rad_mm, float), 30.0))
foot_mm = r_skull * float(np.sin(np.radians(TIPS.half_angle_deg)))
print(f"skull at r~{r_skull:.1f} mm from the target; TIPS half-angle {TIPS.half_angle_deg:.1f} deg "
      f"-> footprint {foot_mm:.1f} mm (the aperture radius, {TIPS.aperture_mm/2:.0f} mm, is "
      f"{TIPS.aperture_mm/2/foot_mm:.0f}x too big here)")

pl = st.place_bowl(tmap, st.BowlConstraints(focal_length_mm=TIPS.roc_mm, bowl_radius_mm=foot_mm,
                                            theta_max_deg=TIPS.acceptance_angle_deg))
print(f"window {np.round(pl.window_center_mni_mm, 2)} mm | incidence {pl.incidence_deg:.1f} deg | "
      f"{pl.n_footprint_patches} patches in the footprint")

### 5 · (optional) Transcranial vs free field

Two forward solves on one grid — through the skull, then with the skull replaced by water — give the insertion loss **at the target**. The literal TIPS cannot be simulated at mouse scale (an 80 mm focal length needs a \~110 mm domain), so the bowl here is *geometrically similar*: the same 35.1° half-angle and f/0.87 at an 18 mm radius of curvature that fits. Skip this cell if you only want the map and the placement.

In [ ]:
RUN_FORWARD = True          # set False to skip (saves ~2 minutes and ~10 GB of scratch)

if RUN_FORWARD:
    from skull_transparency.forward import run_forward_pair
    from skull_transparency.report import forward_peak_volumes

    ROC_MM = 18.0
    APERTURE_MM = 2.0 * ROC_MM * float(np.sin(np.radians(TIPS.half_angle_deg)))
    axis = np.asarray(pl.window_center_mni_mm, float) - TARGET
    axis /= np.linalg.norm(axis)
    apex = TARGET + ROC_MM * axis

    spec_f = TransducerSpec(f0_hz=TIPS.f0_hz, geometry="bowl", roc_mm=ROC_MM,
                            aperture_mm=APERTURE_MM, c0_ms=C_WATER, ppw=TIPS.ppw,
                            acceptance_angle_deg=TIPS.acceptance_angle_deg)
    sim_f = build_brain_center_run(c_map, affine, spec_f, "run_forward", center_phys_mm=TARGET,
                                   bone_threshold=BONE_THRESHOLD, surround_mm=22.0,
                                   input_frame=INPUT_FRAME)
    cmp_ = run_forward_pair(sim_f, out_dir="forward", gpu=0, apex_mm=apex, target_mm=TARGET,
                            roc_mm=ROC_MM, aperture_mm=APERTURE_MM, box_half_mm=3.0, log=print)
    print("\n" + cmp_.summary())
    forward_peak_volumes("forward", out_npz="forward/focal_peaks.npz")   # small artifact
    for _lbl in ("transcranial", "free_field"):                          # reclaim the big traces
        for _f in pathlib.Path("forward", _lbl).glob("*.dat"):
            _f.unlink()

### 6 · The report

The same PDF as `examples/mouse_tips_cerebellum`: placement summary, the target in the skull (orthogonal slices with the Allen cerebellum + a 3-D view inside the semi-transparent skull), the transcranial-vs-free-field comparison if you ran it, the transparency map, incidence, the placement objective, alternative windows, and methods.

In [ ]:
from skull_transparency.report import write_report

rep = write_report(tmap, pl, "report_mouse_tips_cerebellum.pdf",
                   target_name="cerebellum (middle)", bowl_radius_mm=foot_mm,
                   theta_max_deg=TIPS.acceptance_angle_deg,
                   title="Mouse — TIPS 1 MHz on the middle of the cerebellum",
                   bundle=bundle, atlas=ANNOT_NII, atlas_ids=CEREBELLUM_IDS,
                   atlas_label="cerebellum",
                   forward=("forward" if (RUN_FORWARD and pathlib.Path("forward").exists()) else None))
print("wrote", rep)
if IN_COLAB:
    from google.colab import files
    files.download(str(rep))

## Taking it further

- **Another target.** Swap `CEREBELLUM_IDS` for any Allen structure's id set (or set `TARGET` to any coordinate in this skull's frame) and re-run from section 2 — everything downstream is target-agnostic.
- **The propagation movie.** The one section of the example report this notebook leaves out. `examples/mouse_tips_cerebellum/make_propagation_movie.py` re-runs the outward solve with the volume recorder and animates the wave, then `write_report(..., movie=...)` embeds it — playable in Acrobat via LaTeX's `animate`. It writes a few gigabytes of field and needs a TeX installation, so it is better run locally than in Colab.
- **The desktop tools.** The Field Bundle this notebook produces is the durable product: `skull-transparency explore --bundle <dir>` opens it in napari on your own machine, no GPU needed.
- **Your own subject.** The same pipeline takes any sound-speed volume plus its affine — see the [tutorial](https://github.com/pinton-lab/skull_transparency/blob/main/tutorial/tutorial.md). Do not upload identifiable patient data to Colab.